# Halden Energy — Quarter 8: OPEC+ meets in Vienna
**Python notebook (the same data as the workbook)**

OPEC+ has called a meeting to agree a production cut. What it is worth to Halden depends entirely on whether the cut holds. One move in the oil price pushes the parts of Halden in different directions, so the right answer for Halden is not the sum of the right answers for each part.

Unzip the data files next to this notebook before you run it.

In [ ]:
import pandas as pd

outcomes = pd.read_csv('data/opec_outcomes.csv')
before = pd.read_csv('data/before_the_meeting.csv').set_index('parameter')['value']
spread = pd.read_csv('data/how_a_crude_move_spreads.csv').set_index('parameter')['value']
past = pd.read_csv('data/past_cuts.csv')
ahead = pd.read_csv('data/crude_bought_ahead.csv').set_index('parameter')['value']
outcomes

## Part 1 — the worked example

Weight each outcome by its chance, then look at the worst case on its own.

In [ ]:
ex = pd.read_csv('data/worked_example.csv')
ex['oil_fields_per_bbl'] = ex['oil_price_move_usd_bbl'] * spread['oil_fields_per_dollar']
ex['gulf_coast_margin'] = before['gulf_coast_margin_usd_bbl'] + ex['oil_price_move_usd_bbl'] * spread['gulf_coast_margin_per_dollar']
print(ex)
print(f"Expected move for the oil fields: ${(ex['chance'] * ex['oil_fields_per_bbl']).sum():.2f} a barrel")
print('In the down case the oil fields lose $8 a barrel while the refinery gains. Which part of Halden would you rather be?')

## Part 2 — what each outcome does to each part of Halden

In [ ]:
o = outcomes.copy()
move = o['oil_price_move_usd_bbl']
o['oil_fields_per_bbl'] = move * spread['oil_fields_per_dollar']
o['gulf_coast_margin'] = before['gulf_coast_margin_usd_bbl'] + move * spread['gulf_coast_margin_per_dollar']
pump_change_pct = spread['share_reaching_the_pump'] * (move / spread['gallons_per_barrel']) / before['pump_price_usd_gal'] * 100
o['fuel_sales_pct'] = spread['driver_response'] * pump_change_pct
print(o[['label', 'chance_analysts_quote', 'oil_price_usd_bbl', 'oil_fields_per_bbl', 'gulf_coast_margin', 'fuel_sales_pct']].round(3))
print(f"Expected oil price at the analysts' odds: ${(o['chance_analysts_quote'] * o['oil_price_usd_bbl']).sum():.2f}")

## Part 3 — your own odds, from the past cuts

Analysts quote 35 / 40 / 25. Look at what OPEC+ members actually delivered the last five times and decide whether you agree.

In [ ]:
print(past)
print(f"Average share delivered: {past['share_delivered'].mean():.0%}")
# TODO: your own chances for the three outcomes (they must add up to 1)
my_odds = {'full': None, 'partial': None, 'fails': None}

## Part 4 — crude bought ahead

Geneva buys Baton Rouge's crude ahead of the meeting at today's price: 30 days if you plan for the cut to hold, 15 for a partial cut, none if you plan for it to fail. It gains if oil rises and loses if it falls, and the money tied up costs interest either way.

In [ ]:
bbl_day = ahead['baton_rouge_barrels_per_day_at_96pct']
carry_rate = ahead['interest_on_money_tied_up_per_year']
cases = {'plan for the cut to hold': ahead['days_if_planning_for_the_cut_to_hold'],
         'plan for a partial cut': ahead['days_if_planning_for_a_partial_cut'],
         'plan for the cut to fail': ahead['days_if_planning_for_the_cut_to_fail']}
for name, days in cases.items():
    interest = days * bbl_day * before['oil_price_usd_bbl'] * carry_rate / 4 / 1e6
    print(f'{name}: {days:.0f} days, interest ${interest:.1f}M')
    for r in outcomes.itertuples():
        gain = days * bbl_day * r.oil_price_move_usd_bbl / 1e6
        print(f'   {r.label:24s} gain ${gain:7.1f}M   net ${gain - interest:7.1f}M')
    ev = sum(r.chance_analysts_quote * days * bbl_day * r.oil_price_move_usd_bbl / 1e6 for r in outcomes.itertuples()) - interest
    print(f"   expected at the analysts' odds: ${ev:.1f}M")

## Your analysis

Fill in the `# TODO` cells.

In [ ]:
# TODO: redo Part 4 at your own odds from Part 3. Does the answer change?
# TODO: which part of Halden wants the cut to hold, and which is hurt if it holds?
# TODO: what should Geneva plan for? What would you lose in the outcome you're not planning for, and can Halden live with that?
# TODO: given your planning case, what do you do with rigs, run rates and hedges this quarter?

## Your decision

- What does Geneva plan for?
- What does each part of the company do, and why does the company answer differ from the sum of the parts?
- How much of Halden's quarter rides on a meeting in Vienna, and is that the right amount?